<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026_1112Visualization_first_layer_weights_on_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNN の最下位層の特徴検出器 (結合係数) の視覚化

* date: 2026_1112
* filename: `2026_1112Visualization_first_layer_weights_on_CNN.ipynb`
* author: 浅川伸一
* license: MIT
* 元ファイル: `2024notebooks/2021_1029Visualization_the_visual_features_on_CNN.ipynb`

PyTorch (torchvision) 公式の訓練済みモデル 6 種について，入力に最も近い畳み込み層の結合係数 (重み) を画像として表示します。

### 2024 版からの主な変更点
1. 全モデルを `weights='DEFAULT'` (torchvision 公式訓練済み重み) で読み込むよう統一 (`pretrained=True` は廃止された引数)
2. 結合係数の表示方法を修正。旧版は重みを `np.clip(w, 0, 1)` していたため，負の成分がすべて黒になり，フィルタの形がほとんど見えていなかった。本版では各フィルタごとに最小値〜最大値を [0,1] に線形変換して表示する (灰色 = 0 付近，ではなく各フィルタ内の相対値である点に注意)
3. `mobilenet_v2` で `.detach()` が抜けていたため `numpy()` 変換でエラーになっていた点を修正
4. `japanize_matplotlib` は Python 3.12 以降 (現在の Colab) では `distutils` が無いため import に失敗する。図中の表記は英語とし，依存を除去した


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import models

print(f'torch: {torch.__version__}, torchvision: {torchvision.__version__}')

## ResNet18 の構造を確認する

In [ ]:
resnet = models.resnet18(weights='DEFAULT')
resnet.eval()
resnet

上の結果の冒頭は，次のように表示されます

* (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

これは conv1 という名前の層で，

* 入力特徴数: 3 すなわち，入力データはカラー画像なので，赤，緑，青の光の 3 原色のそれぞれを表した数値となっています
* 出力特徴数: 64
* カーネルサイズ: (7, 7) 7 画素 X 7 画素のカーネル幅を持つ畳み込み演算が行われることを示しています
* ストライド: (2, 2) 縦と横とも 2 画素おきにずらしてカーネルが移動することを意味しています。
* パディング: (3, 3) 上下左右に入力画像の周辺に 3 画素分だけ架空の入力領域を確保していることを表します

In [ ]:
w = resnet.conv1.weight.detach()
print(w.size())   # torch.Size([64, 3, 7, 7]) = [出力特徴数, 入力特徴数 (RGB), 縦, 横]
print(f'min={w.min():.3f}, max={w.max():.3f}, mean={w.mean():.3f}')

# 結合係数は正負の値をとるので，[0,1] で切り捨てると負の成分が失われる
fig, axes = plt.subplots(1, 2, figsize=(4, 2))
axes[0].imshow(w[0].permute(1, 2, 0).clip(0, 1).numpy()); axes[0].set_title('clip (old)')
w0 = w[0]; w0 = (w0 - w0.min()) / (w0.max() - w0.min())
axes[1].imshow(w0.permute(1, 2, 0).numpy()); axes[1].set_title('min-max (new)')
for ax in axes: ax.axis('off')
plt.show()

## 表示用の関数

`torchvision.utils.make_grid` で全フィルタを格子状に並べます。
`normalize=True, scale_each=True` により，フィルタごとに最小値〜最大値を [0,1] に変換します。

In [ ]:
def show_filters(weight, title=None, nrow=8, figsize=(5, 5)):
    """畳み込み層の結合係数 [出力, 3, 縦, 横] を RGB 画像として格子状に表示する"""
    weight = weight.detach().cpu()
    assert weight.dim() == 4 and weight.size(1) == 3, f'入力 3 チャンネルの畳み込み層が必要: {tuple(weight.shape)}'
    grid = torchvision.utils.make_grid(weight, nrow=nrow, normalize=True, scale_each=True, padding=1, pad_value=1.0)
    plt.figure(figsize=figsize)
    plt.axis('off')
    plt.imshow(grid.permute(1, 2, 0).numpy(), interpolation='nearest')
    if title is not None:
        plt.title(f'{title}  {tuple(weight.shape)}')
    plt.show()

In [ ]:
# ResNet18
show_filters(resnet.conv1.weight, title='ResNet18 conv1')

In [ ]:
# AlexNet
net = models.alexnet(weights='DEFAULT')
show_filters(net.features[0].weight, title='AlexNet features[0]')

In [ ]:
# Inception v3
net = models.inception_v3(weights='DEFAULT')
show_filters(net.Conv2d_1a_3x3.conv.weight, title='Inception v3 Conv2d_1a_3x3', figsize=(4, 4))

In [ ]:
# VGG11
net = models.vgg11(weights='DEFAULT')
show_filters(net.features[0].weight, title='VGG11 features[0]')

In [ ]:
# DenseNet121
net = models.densenet121(weights='DEFAULT')
show_filters(net.features.conv0.weight, title='DenseNet121 features.conv0')

In [ ]:
# MobileNet v2
net = models.mobilenet_v2(weights='DEFAULT')
show_filters(net.features[0][0].weight, title='MobileNet v2 features[0][0]', figsize=(4, 4))

### 読み取りの注意
* ResNet18 や DenseNet121 に見られる真っ黒のフィルタは，結合係数がほぼ 0 の「使われていない」フィルタです (表示の不具合ではありません)。
* Inception v3, VGG11, MobileNet v2 は 3×3 のカーネルなので，AlexNet (11×11) や ResNet (7×7) のような方位選択性 (ガボールフィルタ様の縞模様) は 1 層目からは読み取りにくくなります。

## 補足: 最初の畳み込み層を自動で探す

モデルごとに層の名前が異なるため，`named_modules()` で最初の `Conv2d` を探す関数を用意しておくと，他のモデルにも流用できます。

In [ ]:
def first_conv(model):
    for name, m in model.named_modules():
        if isinstance(m, torch.nn.Conv2d):
            return name, m
    raise ValueError('Conv2d が見つかりません')

for arch in ['resnet18', 'alexnet', 'inception_v3', 'vgg11', 'densenet121', 'mobilenet_v2']:
    net = models.get_model(arch, weights='DEFAULT')
    name, conv = first_conv(net)
    print(f'{arch:14s} {name:24s} {tuple(conv.weight.shape)}')